In [1]:
# 0) SETUP: imports, download/leitura dos dados, preparação e split

import sys
import subprocess
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

# Tentar ler direto da URL; se preferir, use !wget no shell do notebook
URL = "https://raw.githubusercontent.com/alexeygrigorev/datasets/master/car_fuel_efficiency.csv"
df = pd.read_csv(URL)

# 1) Preenchimento de valores ausentes com zero
df = df.fillna(0)

# 2) Definir target e features
target_col = 'fuel_efficiency_mpg'
y = df[target_col].values
df_features = df.drop(columns=[target_col])

# 3) Split 60/20/20 (com random_state=1)
df_full_train, df_test, y_full_train, y_test = train_test_split(
    df_features, y, test_size=0.2, random_state=1
)
df_train, df_val, y_train, y_val = train_test_split(
    df_full_train, y_full_train, test_size=0.25, random_state=1
)  # 0.25 de 0.8 => 0.2

# 4) DictVectorizer (sparse=True)
dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient='records')
val_dicts   = df_val.to_dict(orient='records')
test_dicts  = df_test.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val   = dv.transform(val_dicts)
X_test  = dv.transform(test_dicts)

feature_names = dv.get_feature_names_out()

print("Tamanhos:")
print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Ex. de features:", feature_names[:10])


Tamanhos:
Train: (5822, 14) Val: (1941, 14) Test: (1941, 14)
Ex. de features: ['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'engine_displacement' 'fuel_type=Diesel'
 'fuel_type=Gasoline' 'horsepower' 'model_year' 'num_cylinders'
 'num_doors']


In [10]:
df_features

,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,model_year,origin,fuel_type,drivetrain,num_doors
0,170,3.0,159.0,3413.433759,17.7,2003,Europe,Gasoline,All-wheel drive,0.0
1,130,5.0,97.0,3149.664934,17.8,2007,USA,Gasoline,Front-wheel drive,0.0
2,170,0.0,78.0,3079.038997,15.1,2018,Europe,Gasoline,Front-wheel drive,0.0
3,220,4.0,0.0,2542.392402,20.2,2009,USA,Diesel,All-wheel drive,2.0
4,210,1.0,140.0,3460.870990,14.4,2009,Europe,Gasoline,All-wheel drive,2.0
...,...,...,...,...,...,...,...,...,...,...
9699,140,5.0,164.0,2981.107371,17.3,2013,Europe,Diesel,Front-wheel drive,0.0
9700,180,0.0,154.0,2439.525729,15.0,2004,USA,Gasoline,All-wheel drive,0.0
9701,220,2.0,138.0,2583.471318,15.1,2008,USA,Diesel,All-wheel drive,-1.0
9702,230,4.0,177.0,2905.527390,19.4,2011,USA,Diesel,Front-wheel drive,1.0


In [11]:
y

array([13.23172891, 13.68821744, 14.246341  , ..., 17.18658678,
       15.33155059, 14.8844674 ], shape=(9704,))

In [2]:
# Q1: DecisionTreeRegressor com max_depth=1 e feature do primeiro split

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Índice da feature usada na raiz (nó 0):
root_feature_idx = dt.tree_.feature[0]
split_feature = feature_names[root_feature_idx]

# Para comparar com as alternativas, vamos pegar o "nome base"
# (para variáveis categóricas one-hot, remove o sufixo "=valor")
split_base = split_feature.split('=')[0]

print("Feature de split (nome completo):", split_feature)
print("Feature base:", split_base)

# Opções do enunciado
opcoes = ['vehicle_weight', 'model_year', 'origin', 'fuel_type']

def escolhe_alternativa(nome_base):
    # Para 'origin' e 'fuel_type', se qualquer dummy gerou o split, respondemos pelo nome base
    # Para numéricas, o nome já coincide
    # Se não bater exatamente, escolhemos a 'mais próxima' por regra simples
    if nome_base in opcoes:
        return nome_base
    # Heurística: se começa com "origin=" ou "fuel_type=", mapeia para base
    if split_feature.startswith('origin='):
        return 'origin'
    if split_feature.startswith('fuel_type='):
        return 'fuel_type'
    # Caso contrário, procura a mais "similar"
    from difflib import get_close_matches
    m = get_close_matches(nome_base, opcoes, n=1)
    return m[0] if m else nome_base

print("Resposta (alternativa):", escolhe_alternativa(split_base))


Feature de split (nome completo): vehicle_weight
Feature base: vehicle_weight
Resposta (alternativa): vehicle_weight


In [3]:
# Q2: RandomForestRegressor com n_estimators=10, random_state=1, n_jobs=-1

from sklearn.ensemble import RandomForestRegressor
from math import sqrt

rf10 = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf10.fit(X_train, y_train)

y_pred_val = rf10.predict(X_val)
rmse_val = sqrt(mean_squared_error(y_val, y_pred_val))

print("RMSE validação:", rmse_val)

# Escolher alternativa mais próxima
alts = [0.045, 0.45, 4.5, 45.0]
alt_escolhida = min(alts, key=lambda a: abs(a - rmse_val))
print("Resposta (alternativa mais próxima):", alt_escolhida)

RMSE validação: 0.4595777223092726
Resposta (alternativa mais próxima): 0.45


In [4]:
# Q3: Loop em n_estimators de 10 a 200, step 10, random_state=1
# Encontrar o primeiro n em que o RMSE (arredondado a 3 casas) não melhora mais

results = []
best_rmse_rounded = None
stop_after = None

for n in range(10, 201, 10):
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))
    rmse3 = round(rmse, 3)
    results.append((n, rmse, rmse3))
    if best_rmse_rounded is None or rmse3 < best_rmse_rounded:
        best_rmse_rounded = rmse3
        stop_after = n  # por enquanto, melhor
    # Se rmse3 ficar igual ao melhor e nunca mais diminuir, mantemos o primeiro n com o melhor

# Mostrar tabela
print("n_estimators | RMSE | RMSE(3 casas)")
for n, r, r3 in results:
    print(f"{n:12d} | {r:,.6f} | {r3:.3f}")

print("\nMelhor RMSE (3 casas):", best_rmse_rounded, "atingido em n_estimators =", stop_after)

# De acordo com o enunciado, se nunca parar de melhorar, usamos a última iteração (200).
# Aqui 'stop_after' já representa o 1º n onde o melhor arredondado foi atingido.

# Escolher alternativa mais próxima dentre: 10, 25, 80, 200
opcoes = [10, 25, 80, 200]
alt = min(opcoes, key=lambda x: abs(x - stop_after))
print("Resposta (alternativa mais próxima):", alt)


n_estimators | RMSE | RMSE(3 casas)
          10 | 0.459578 | 0.460
          20 | 0.453591 | 0.454
          30 | 0.451687 | 0.452
          40 | 0.448721 | 0.449
          50 | 0.446657 | 0.447
          60 | 0.445460 | 0.445
          70 | 0.445126 | 0.445
          80 | 0.444984 | 0.445
          90 | 0.444861 | 0.445
         100 | 0.444652 | 0.445
         110 | 0.443579 | 0.444
         120 | 0.443912 | 0.444
         130 | 0.443703 | 0.444
         140 | 0.443355 | 0.443
         150 | 0.442898 | 0.443
         160 | 0.442761 | 0.443
         170 | 0.442801 | 0.443
         180 | 0.442362 | 0.442
         190 | 0.442494 | 0.442
         200 | 0.442479 | 0.442

Melhor RMSE (3 casas): 0.442 atingido em n_estimators = 180
Resposta (alternativa mais próxima): 200


In [5]:
# Q4: Para cada max_depth em [10, 15, 20, 25], calcular média do RMSE
# em n_estimators = 10..200 (passo 10). random_state=1

depths = [10, 15, 20, 25]
media_por_depth = {}

for d in depths:
    rmses = []
    for n in range(10, 201, 10):
        rf = RandomForestRegressor(n_estimators=n, max_depth=d, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, pred))
        rmses.append(rmse)
    media_rmse = float(np.mean(rmses))
    media_por_depth[d] = media_rmse

print("Médias de RMSE por max_depth:")
for d in depths:
    print(f"max_depth={d}: média RMSE = {media_por_depth[d]:.6f}")

best_depth = min(media_por_depth, key=media_por_depth.get)
print("Melhor max_depth pela média do RMSE:", best_depth)

# Escolher alternativa exata (10, 15, 20, 25)
print("Resposta (alternativa):", best_depth)


Médias de RMSE por max_depth:
max_depth=10: média RMSE = 0.441808
max_depth=15: média RMSE = 0.445417
max_depth=20: média RMSE = 0.446253
max_depth=25: média RMSE = 0.445910
Melhor max_depth pela média do RMSE: 10
Resposta (alternativa): 10


In [6]:
# Q5: Treinar RF com n_estimators=10, max_depth=20, random_state=1
# Extrair feature_importances_ e agregar por base (para o caso de dummies)
# Depois comparar apenas entre: vehicle_weight, horsepower, acceleration, engine_displacement

rf_imp = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf_imp.fit(X_train, y_train)

importances = rf_imp.feature_importances_
names = feature_names

# Agregar importâncias por "nome base" (antes do '='):
from collections import defaultdict
agg = defaultdict(float)
for name, imp in zip(names, importances):
    base = name.split('=')[0]
    agg[base] += imp

candidatas = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']

print("Importâncias agregadas (algumas):")
for k in sorted(agg, key=agg.get, reverse=True)[:15]:
    print(f"{k}: {agg[k]:.6f}")

# Filtrar para as 4 pedidas
subset = {k: agg.get(k, 0.0) for k in candidatas}
print("\nEntre as 4 candidatas:")
for k, v in subset.items():
    print(f"{k}: {v:.6f}")

mais_importante = max(subset, key=subset.get)
print("\nResposta (mais importante entre as 4):", mais_importante)


Importâncias agregadas (algumas):
vehicle_weight: 0.959150
horsepower: 0.015998
acceleration: 0.011480
engine_displacement: 0.003273
model_year: 0.003212
num_cylinders: 0.002343
num_doors: 0.001635
origin: 0.001521
drivetrain: 0.000702
fuel_type: 0.000686

Entre as 4 candidatas:
vehicle_weight: 0.959150
horsepower: 0.015998
acceleration: 0.011480
engine_displacement: 0.003273

Resposta (mais importante entre as 4): vehicle_weight


In [7]:
# Q6: XGBoost (100 rounds), comparar eta=0.3 vs 0.1 no RMSE de validação

# Instalar xgboost se necessário
try:
    import xgboost as xgb
except ImportError:
    print("Instalando xgboost...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    import xgboost as xgb

# DMatrix para treino e validação
dtrain = xgb.DMatrix(X_train, label=y_train)
dval   = xgb.DMatrix(X_val,   label=y_val)

watchlist = [(dtrain, 'train'), (dval, 'eval')]

def treina_avalia(eta_value):
    params = {
        'eta': eta_value,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    booster = xgb.train(
        params=params,
        dtrain=dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )
    pred_val = booster.predict(dval)
    rmse = float(np.sqrt(mean_squared_error(y_val, pred_val)))
    return rmse

rmse_03 = treina_avalia(0.3)
rmse_01 = treina_avalia(0.1)

print(f"RMSE (eta=0.3): {rmse_03:.6f}")
print(f"RMSE (eta=0.1): {rmse_01:.6f}")

if abs(rmse_03 - rmse_01) < 1e-9:
    resposta = "Both give equal value"
else:
    resposta = "0.3" if rmse_03 < rmse_01 else "0.1"

print("Resposta:", resposta)


Instalando xgboost...



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python -m pip install --upgrade pip


RMSE (eta=0.3): 0.450178
RMSE (eta=0.1): 0.426228
Resposta: 0.1
